# Training Model Market Value Category

Notebook ini menjalankan Task 3 saja: time based split, hybrid sampling hanya untuk train, training tiga model, evaluasi, dan penyimpanan artifact.

Input:
- `data/model/players_model.csv`

Output utama:
- `models/logistic_regression.pkl`
- `models/random_forest.pkl`
- `models/xgboost.pkl`
- `models/best_model.pkl`
- `models/preprocessor.pkl`
- `models/label_encoder.pkl`
- `data/output/model_metrics.csv`
- `data/output/classification_report.csv`
- `data/output/confusion_matrix.csv`
- `data/output/feature_importance.csv`
- `data/output/label_distribution_before_after_sampling.csv`


In [1]:
import sys

!{sys.executable} -m pip install matplotlib imbalanced-learn xgboost


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Import

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling.train import MODEL_DATA_FILE, train_and_evaluate

print(f'Project root: {PROJECT_ROOT}')
print(f'Model data  : {MODEL_DATA_FILE}')

Project root: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL
Model data  : C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\model\players_model.csv


## 2. Train dan Evaluate

In [3]:
summary, metrics_df = train_and_evaluate()

print('Training selesai.')
print(f'Best model: {summary["best_model"]}')
print(f'Best scenario: {summary["best_scenario"]}')
print(f'Best params: {summary["best_params"]}')
print(f'XGBoost available: {summary["xgboost_available"]}')
print(f'Train rows: {summary["train_rows"]}')
print(f'Validation rows: {summary["validation_rows"]}')
print(f'Test rows: {summary["test_rows"]}')


Training selesai.
Best model: xgboost
Best scenario: hybrid_sampling_light
Best params: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'reg_lambda': 1}
XGBoost available: True
Train rows: 6138
Validation rows: 1334
Test rows: 2739


## 3. Metrics

In [4]:
print('Top validation metrics:')
display(
    metrics_df[metrics_df['split'] == 'validation']
    .sort_values('macro_f1', ascending=False)
    .head(20)
)

print('Test metrics best model only:')
display(metrics_df[metrics_df['split'] == 'test'])


Top validation metrics:


,model,scenario,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,recall_tinggi,params,selected_for_test
0,xgboost,hybrid_sampling_light,validation,0.709895,0.708318,0.729336,0.713180,0.707294,0.752381,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",False
1,xgboost,hybrid_sampling_light,validation,0.703898,0.705029,0.723261,0.708244,0.700870,0.742857,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",False
2,xgboost,hybrid_sampling_light,validation,0.706897,0.706656,0.718695,0.708196,0.704420,0.714286,"{'n_estimators': 500, 'max_depth': 3, 'learnin...",False
3,xgboost,hybrid_sampling_light,validation,0.705397,0.704459,0.718553,0.706958,0.702864,0.719048,"{'n_estimators': 300, 'max_depth': 4, 'learnin...",False
4,xgboost,hybrid_sampling_light,validation,0.703148,0.701083,0.723963,0.706831,0.700449,0.752381,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",False
5,random_forest,hybrid_sampling_light,validation,0.699400,0.709363,0.710805,0.705754,0.697656,0.709524,"{'n_estimators': 500, 'max_depth': 15, 'min_sa...",False
6,random_forest,hybrid_sampling_light,validation,0.698651,0.708902,0.707959,0.704263,0.697020,0.700000,"{'n_estimators': 300, 'max_depth': 15, 'min_sa...",False
7,xgboost,hybrid_sampling_light,validation,0.701649,0.703786,0.712205,0.703841,0.699322,0.704762,"{'n_estimators': 500, 'max_depth': 4, 'learnin...",False
8,xgboost,class_weight_balanced,validation,0.703148,0.700502,0.715937,0.703558,0.700523,0.714286,"{'n_estimators': 300, 'max_depth': 4, 'learnin...",False
9,random_forest,hybrid_sampling_light,validation,0.697901,0.705780,0.707960,0.703278,0.696319,0.704762,"{'n_estimators': 300, 'max_depth': None, 'min_...",False


Test metrics best model only:


,model,scenario,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,recall_tinggi,params,selected_for_test
189,xgboost,hybrid_sampling_light,test,0.682001,0.679955,0.715627,0.688294,0.677779,0.778495,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",True


## 4. Validasi Split dan Sampling

In [5]:
print('Train label original:', summary['train_label'])
print('Train label used by best scenario:', summary['best_train_label'])
print('Validation label original:', summary['validation_label'])
print('Test label original:', summary['test_label'])

assert summary['validation_rows'] > 0
assert summary['test_rows'] > 0
assert summary['best_scenario'] in {'no_sampling', 'class_weight_balanced', 'hybrid_sampling_light'}

print('Confusion matrix best model:')
print(summary['confusion_matrix'])
print()
print('Top feature importance best model:')
display(summary['feature_importance'].head(20))
print('Validasi split, skenario, dan output best model lulus.')


Train label original: {'Menengah': 2868, 'Rendah': 2319, 'Tinggi': 951}
Train label used by best scenario: {'Menengah': 2500, 'Rendah': 2500, 'Tinggi': 2500}
Validation label original: {'Menengah': 648, 'Rendah': 476, 'Tinggi': 210}
Test label original: {'Menengah': 1342, 'Rendah': 932, 'Tinggi': 465}
Confusion matrix best model:
[[751 164  17]
 [411 755 176]
 [ 14  89 362]]

Top feature importance best model:


,feature,importance
0,prev_mv_category_Tinggi,0.156036
1,prev_mv_category_Menengah,0.064844
2,prev_season_mv,0.057871
3,prev_season_mv_log,0.056778
4,prev_mv_distance_to_10,0.038722
5,prev_mv_category_Rendah,0.031574
6,prev_mv_distance_to_30,0.023952
7,club_total_mv_log,0.023469
8,club_total_mv_mio,0.021359
9,club_total_mv_rank_league_season,0.012976


Validasi split, skenario, dan output best model lulus.
